# 04 - Levitate on one tile (closed loop)

**Goal:** see the control loop work: sensor -> PD controller -> current allocation -> coils -> mover.

The controller is set by a *bandwidth* in Hz (`control.bandwidth_Hz`): higher = stiffer, but
needs more current and a faster, cleaner sensor. v1 simulates x, y, z motion; tilt torques are
commanded to zero and logged.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from maglev_sim import *
from maglev_sim import plotting

In [ ]:
cfg = load_config("../configs/examples/single_tile_quick.yaml")
s = build_system(cfg)
p0 = np.r_[s.grid.tile_center(0), s.gap]
res = simulate(s, Trajectory.hold(p0), duration=0.5)   # first run builds the fast force model (~10 s)
print(res.summary())
plotting.plot_simulation(res);

## A small move, then a disturbance: drop the gap target by 0.5 mm

In [ ]:
p1 = p0 + [0.015, 0.010, 0.0]
traj = Trajectory([(0, p0), (0.1, p0), (0.5, p1), (0.7, p1), (0.8, p1 - [0, 0, 0.0005])])
res = simulate(s, traj, duration=1.0)
print(res.summary())
plotting.plot_simulation(res);

## Effect of controller bandwidth and sensor noise

In [ ]:
rows = []
for bw in [4, 8, 16, 30]:
    for noise in [0.005, 0.02, 0.05]:
        c = with_changes(cfg, {"control.bandwidth_Hz": [bw, bw, bw * 1.5], "simulation.sensor_noise_mm": noise})
        r = simulate(build_system(c), Trajectory.move(p0, p1, 0.1, 0.4), duration=0.7)
        rows.append({"bandwidth_Hz": bw, "noise_mm": noise, **{k: r.info[k] for k in
                     ["rms_error_mm", "peak_current_A", "mean_power_W", "touchdown"]}})
import pandas as pd
pd.DataFrame(rows).round(3)

**What to look for:** noisy sensors + high bandwidth = jittery currents (heat, EMI into Hall
sensors). This tells the electronics subteam what sensor noise is acceptable.